# 05 EWMA Volatility Model

Calculate the lagged volatility matrix used by the synthetic options. At date t the variance includes returns only through t minus one.

Run cells from top to bottom, or choose **Run All** for this notebook only. Each module saves its outputs for the next notebook. Restart the kernel after pulling code changes.


## 1. Imports and saved run


In [ ]:
%matplotlib inline
from pathlib import Path
import sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src' / 'research_config.py').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Open Jupyter inside the Pairs_trading repository.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from src.notebook_session import NotebookSession

session = NotebookSession.active()
cfg = session.config
RUN_DIR = session.run
session.begin('05_volatility', ['04_signal', '01_rf'])


## 2. Load formation and test prices

The same EWMA implementation is used inside the daily backtest.


In [ ]:
from src.backtest import precompute_oos_ewma_volatility
train_prices = session.frame('train_prices')
test_prices = session.frame('test_prices')
display(pd.Series({'ewma_lambda': cfg.ewma_lambda, 'formation_sessions': len(train_prices), 'test_sessions': len(test_prices)}))


## 3. Calculate and check volatility

Save all dates so Module 06 can inspect the actual next-close input for a snapshot signal.


In [ ]:
volatility = precompute_oos_ewma_volatility(train_prices, test_prices, lambda_=cfg.ewma_lambda)
assert np.isfinite(volatility.to_numpy()).all() and (volatility > 0).all().all()
session.save('oos_ewma_volatility', volatility)
display(volatility.head())
volatility.iloc[:, :4].plot(figsize=(10, 4), title='Lagged annualized EWMA volatility')
plt.show()


## Save module completion

Wait for this confirmation before moving to the next notebook.


In [ ]:
session.finish('05_volatility')
